In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from typing import Optional, List
from pydantic import BaseModel, Field
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
# We load the CV in pdf format
loader = PyPDFLoader("./CV_Template_Abraham_Traore.pdf") #/Users/abrahamtraore/Desktop 
docs = loader.load() # The content of the CV is stored in CV

In [5]:
docs

[Document(metadata={'producer': 'ilovepdf.com', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2018-07-13T11:14:25+02:00', 'source': './CV_Template_Abraham_Traore.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Forename SURNAME \ne-mail: professional email address tel: UK landline or mobile \nEducation and Qualifications   \n2000-2003 University/Universities  Degree and Subject  \n Location; City and Country  applicable additional info\n   \nWork Experience \n \nSep-07 – Aug-10 Official Company Name  City, Country \nJob title  \n\uf0b7 Please use 3-4 bullets maximum to describe your job function & \nresponsibilities  \n\uf0b7 Concentrate on your achievements, and what you have distinctly \ncontributed to in each role, using quantitative examples where possible \n\uf0b7 Examples that may assist you –  \n\uf0b7 “Advised client’s Digital Media division on £3M international expansion, \ncoordinating a team of 8 analysts during initial research phase” \n\uf0b7 “Str

In [6]:
# Splitting of the text into chunks, the size of a chunk being 1000
text_split = RecursiveCharacterTextSplitter(chunk_size=1000)
splits = text_split.split_documents(docs)

In [7]:
splits

[Document(metadata={'producer': 'ilovepdf.com', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2018-07-13T11:14:25+02:00', 'source': './CV_Template_Abraham_Traore.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Forename SURNAME \ne-mail: professional email address tel: UK landline or mobile \nEducation and Qualifications   \n2000-2003 University/Universities  Degree and Subject  \n Location; City and Country  applicable additional info\n   \nWork Experience \n \nSep-07 – Aug-10 Official Company Name  City, Country \nJob title  \n\uf0b7 Please use 3-4 bullets maximum to describe your job function & \nresponsibilities  \n\uf0b7 Concentrate on your achievements, and what you have distinctly \ncontributed to in each role, using quantitative examples where possible \n\uf0b7 Examples that may assist you –  \n\uf0b7 “Advised client’s Digital Media division on £3M international expansion, \ncoordinating a team of 8 analysts during initial research phase” \n\uf0b7 “Str

In [10]:
# Computation of the vectors associated with the vectors --> this is essential because statistical techniques work only 
# on numbers. 
# To turn numbers into vectors, we use embeddings. For this specific example, we use an embedding strategy from Hugginface. It is worth 
# that other strategies exist.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") # Choice of the embedding method
vectorstore = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings) # Langchain method for the computation of the vectors associated to chunks
retriever = vectorstore.as_retriever(search_type="similarity") # Definition of the strategy for retrieval, in this case, the retrieval relies on the cosine similarity.

/opt/anaconda3/envs/test_virtual_environment/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 8450.14it/s]


In [11]:
def get_answers_to_a_query(retriever, queries) :
    """
    Searches for relevant chunks for each provided query.
    """
    outputs = []
    for query in queries: # for every query
        docs = retriever.invoke(query) # We compute the answer of a specific query: this method recovers the query, apply the embedding method to turn it into a vector and recover the closest answer, the proximity being measured by the cosine similarity
        outputs.extend([doc.page_content for doc in docs]) # Storage of the output of the query
    return list(set(outputs))

In [12]:
queries = [
    f"The name of the candidate",
    f"The main job role of the candidate",
    f"Total candidate's years of experience",
    "The candidate's bachelor degree",
    "The candidate's years of experience working with python",
    "The candidate's github url",
    "The candidate's linkedin url",
    "The candidate's years of experience working on machine learning projects"
]
relevant_chunks = get_answers_to_a_query(retriever, queries)

In [13]:
print(relevant_chunks)

['Job title \n\uf0b7 Use past tense for roles you have completed \n\uf0b7 Please set dates using the abbreviated month and two digits for the year, \nyou must include months as well as years  \n\uf0b7 Make sure your CV is an accurate reflection of you and what you want to \nhighlight about your experience  \n\uf0b7 Stick to facts you can easily discuss. Avoid subjective comments \n  \nAdditional Information \n \nInterests: Concentrate on activities you participate in and are willing to talk about. You \nshould highlight achievements in those activities. Eg. rather than just listing \n‘running’ say ‘running – participated in several marathons, President of the \nOxford Runners Club’ \n \nAchievements: List academic or other achievements here, for example  \n First Class Honours, Previous University  \n Study abroad scholarship (selected 3 out of 600 students) \n   Principal Cellist of London Youth Orchestra \n  \nNationality: your nationality, dual nationality, and any additional work a